In [2]:
!pip install ai_edge_litert

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.0/21.0 MB 63.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [ai_edge_litert]


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image, ImageOps
from ai_edge_litert.interpreter import Interpreter

In [8]:
MODEL = "mn.tflite"
PHOTOS = ["/content/superman.jpeg", "/content/im.jpg"]
SORT_BY = "rsrrMale"

NAMES = ["rsrrFemale", "rsrrMale", "nudity",
         "underage", "face", "fullBody"]


net = Interpreter(model_path=MODEL, num_threads=4)
net.allocate_tensors()

inp = net.get_input_details()[0]
outs = net.get_output_details()
h, w = map(int, inp["shape"][1:3])

score_id = next(o["index"] for o in outs if list(o["shape"]) == [1, 6])
embed_id = next(o["index"] for o in outs if list(o["shape"]) == [1, 512])

def predict(path):
    with Image.open(path) as img:
        img = ImageOps.fit(
            ImageOps.exif_transpose(img).convert("RGB"),
            (w, h), method=Image.Resampling.BILINEAR
        )
        x = np.asarray(img)

    if inp["dtype"] == np.float32:
        x = x.astype(np.float32) / 255
    else:
        x = x.astype(inp["dtype"])

    net.set_tensor(inp["index"], x[None])
    net.invoke()
    return (net.get_tensor(score_id)[0],
            net.get_tensor(embed_id)[0])


rows, embeddings = [], []

for path in map(Path, PHOTOS):
    if not path.is_file():
        print("Не найден:", path)
        continue

    scores, embedding = predict(path)
    rows.append({"photo": path.name, **dict(zip(NAMES, map(float, scores)))})
    embeddings.append(embedding)

df = pd.DataFrame(rows)

if not df.empty:
    order = np.argsort(df[SORT_BY].to_numpy())[::-1]
    df = df.iloc[order].reset_index(drop=True)

    display(df.round(5))
    df.to_csv("tinder_scores.csv", index=False)
    np.save("tinder_embeddings.npy", np.stack([embeddings[i] for i in order]))

,photo,rsrrFemale,rsrrMale,nudity,underage,face,fullBody
0,im.jpg,0.15535,0.45268,0.00049,0.11601,0.99751,0.97188
1,superman.jpeg,0.07807,0.38490,0.11830,0.00699,0.97484,0.98941
